In [44]:
import torch
from torch import nn
from torch.nn import functional as F

import random
import math
import re
import time

In [45]:

num_digits = 3

dataset_size = 64_000
train_proportion = 0.9

In [46]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Step 1: Construct a tokenizer

In [47]:
pad_token="[PAD]"
eos_token="[EOS]"

In [48]:
class character_level_tokenizer:
    """
    character-level
    """
    def __init__(self):
        self.vocab = [str(x) for x in range(10)] + ["+", "="] + [pad_token, eos_token]
        self.token_to_id = {v : k for k, v in enumerate(self.vocab)}
        self.id_to_token = {k : v for k, v in enumerate(self.vocab)}
        self.ntokens = len(self.vocab)
        self.pattern = f"[^{re.escape(''.join(self.vocab))}]"

    def clean(self, text):
        """
        removes all characters not in the vocabulary
        """
        out = re.sub(self.pattern, "", text)
        return out

    def pre_tokenization(self, text):
        """
        character-level
        """
        return [c for c in text]

    def encode(self, text):
        text_list = self.pre_tokenization(self.clean(text))
        return [self.token_to_id[c] for c in text_list]

    def decode(self, token_list):
        return "".join([self.id_to_token[x] for x in token_list])

In [49]:
tokenizer = character_level_tokenizer()
ntokens = tokenizer.ntokens
ntokens

14

In [50]:

prompt = "12 + 42 ="
inputs = tokenizer.encode(prompt)
inputs, tokenizer.decode(inputs)

([1, 2, 10, 4, 2, 11], '12+42=')

## Step 2: Create a dataset for arithmetic operations

In [51]:

def sample_datapoint(num_digits = 3):
    a_list = [random.randint(0, 9) for _ in range(num_digits)]
    b_list = [random.randint(0, 9) for _ in range(num_digits)]
    a_int = int("".join([str(x) for x in a_list]))
    b_int = int("".join([str(x) for x in b_list]))
    a_str = "".join([str(x) for x in a_list])
    b_str = "".join([str(x) for x in b_list])
    sum_int = a_int + b_int
    return (a_str + "+" + b_str + "=", str(sum_int))

sample_datapoint(3)

('617+135=', '752')

In [52]:
data = []
for _ in range(dataset_size):
    data.append(sample_datapoint(num_digits))
data[:4]

[('485+688=', '1173'),
 ('802+812=', '1614'),
 ('747+728=', '1475'),
 ('813+628=', '1441')]

In [53]:

data_train = data[: int(train_proportion * dataset_size)]
data_test = data[int(train_proportion * dataset_size):]

len(data_train),len(data_test)

(57600, 6400)

## Step 3: Construct a model

In [54]:
class PositionalEncoding(nn.Module):
    r"""Inject some information about the relative or absolute position of the tokens in the sequence.
        The positional encodings have the same dimension as the embeddings, so that the two can be summed.
        Here, we use sine and cosine functions of different frequencies.
    .. math:
        \text{PosEncoder}(pos, 2i) = sin(pos/10000^(2i/d_model))
        \text{PosEncoder}(pos, 2i+1) = cos(pos/10000^(2i/d_model))
        \text{where pos is the word position and i is the embed idx)
    Args:
        d_model: the embed dim (required).
        dropout: the dropout value (default=0.1).
        max_len: the max. length of the incoming sequence (default=5000).
    """

    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        r"""Inputs of forward function
        Args:
            x: the sequence fed to the positional encoder model (required).
        Shape:
            x: [sequence length, batch size, embed dim]
            output: [sequence length, batch size, embed dim]
        """

        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

In [55]:
class TransformerModel(nn.Transformer):
    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):
        super(TransformerModel, self).__init__(d_model=ninp,
                                               nhead=nhead,
                                               dim_feedforward=nhid,
                                               num_encoder_layers=nlayers)
        self.input_emb = nn.Embedding(ntoken, ninp)
        self.pos_encoder = PositionalEncoding(ninp, dropout)
        self.decoder = nn.Linear(ninp, ntoken)

        self.ninp = ninp
        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        nn.init.uniform_(self.input_emb.weight, -initrange, initrange)
        nn.init.zeros_(self.decoder.bias)
        nn.init.uniform_(self.decoder.weight, -initrange, initrange)

    def _generate_square_subsequent_mask(self, sz):
        return torch.log(torch.tril(torch.ones(sz,sz)))

    def forward(self, src):
        mask = self._generate_square_subsequent_mask(len(src)).to(device)
        self.src_mask = mask

        src = self.input_emb(src) * math.sqrt(self.ninp)
        src = self.pos_encoder(src)
        output_enc = self.encoder(src, mask=self.src_mask)
        output_dec = self.decoder(output_enc)
        return F.log_softmax(output_dec, dim=-1), output_enc

In [56]:
model = TransformerModel(ntoken = ntokens,
                         ninp = 128,
                         nhead = 16,
                         nhid = 64,
                         nlayers = 8)
model.to(device)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


TransformerModel(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-7): 8 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): Linear(in_features=128, out_features=14, bias=True)
  (input_emb): Embedding(14, 128)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [57]:
print("number of parameters: {}".format(sum([x.numel() for x in model.parameters()])))


number of parameters: 668942


### Useful functions


In [58]:
def generate(model, prompts, new_tokens = 5, mode = "greedy", num_samples = 1, temperature = 0.8):
    input_tensor = torch.repeat_interleave(prompts, repeats = num_samples, dim = 1).to(device)
    # (prompt_length, batch_size * num_samples)
    for _ in range(new_tokens):
        output, _ = model(input_tensor) # (prompt_length, batch_size * num_samples, ntokens)
        logits = output[-1,:,:] # (batch_size * num_samples, ntokens)
        if mode == "greedy":
            tokens = torch.argmax(logits, -1).view((1,-1)) # (1, batch_size * num_samples)
        else: # mode == "sampling"
            logits /= temperature
            probs = torch.softmax(logits, dim=-1)
            tokens = torch.multinomial(probs, num_samples = 1).view((1,-1)) # (1, batch_size * num_samples)
        input_tensor = torch.cat((input_tensor, tokens), 0)
    return input_tensor

In [59]:
model.eval()

prompt = "2+3="
prompt_tensor = torch.tensor(tokenizer.encode(prompt)).view((-1,1))
output = generate(model, prompt_tensor).view((1,-1))
output, tokenizer.decode(output[0].tolist())

(tensor([[ 2, 10,  3, 11,  1,  5,  5,  5,  5]], device='cuda:0'), '2+3=15555')

In [60]:
def pad(token_list, type_list = "prompts"):
    max_length = max([len(x) for x in token_list])
    out = []
    for x in token_list:
        if type_list == "prompts":
            out.append([tokenizer.token_to_id[pad_token]] * (max_length - len(x)) + x)
        if type_list == "answers":
            out.append(x + [tokenizer.token_to_id[eos_token]] + [tokenizer.token_to_id[pad_token]] * (max_length - len(x)))
    return out, max_length

In [61]:
prompts = [tokenizer.encode("1+1="), tokenizer.encode("21+35=")]
answers = [tokenizer.encode("2"), tokenizer.encode("56")]
padded_prompts, _ = pad(prompts, "prompts")
padded_answers, _ = pad(answers, "answers")
padded_prompts, padded_answers
[tokenizer.decode(p) for p in padded_prompts], [tokenizer.decode(p) for p in padded_answers]

(['[PAD][PAD]1+1=', '21+35='], ['2[EOS][PAD]', '56[EOS]'])

In [62]:
def get_batch(split, i, batch_size):
    data = data_train if split == 'train' else data_test

    prompts = [data[i][0] for i in range(i, i + batch_size)]
    encoded_prompts = [tokenizer.encode(prompt) for prompt in prompts]
    padded_prompts, prompt_length = pad(encoded_prompts, "prompts")

    answers = [data[i][1] for i in range(i, i + batch_size)]
    encoded_answers = [tokenizer.encode(answer) for answer in answers]
    padded_answers, answers_length = pad(encoded_answers, "answers")

    X = torch.stack([torch.tensor(x) for x in padded_prompts], 1)
    Y = torch.stack([torch.tensor(x) for x in padded_answers], 1)
    return X, Y, prompt_length, answers_length, prompts, answers

In [63]:

X, Y, prompt_length, answers_length, prompts, answers = get_batch("train", 43, 16)
X.shape, Y.shape, prompt_length, answers_length, prompts[0], answers[0]

(torch.Size([8, 16]), torch.Size([5, 16]), 8, 4, '985+466=', '1451')

## Step 4: Evaluate

In [64]:

batch_size = 16

In [65]:
def evaluate(batch_size = batch_size):
    # Turn on evaluation mode disables dropout.
    model.eval()
    correct = 0.
    with torch.no_grad():
        for batch, i in enumerate(range(0, len(data_test) - 1, batch_size)):
            prompts, target_answers, prompt_length, answers_length, _, _ = get_batch("test", i, batch_size)
            prompts = prompts.to(device) # (prompt_length, batch_size)
            target_answers = target_answers.to(device) # (answers_length + 1, batch_size)
            output = generate(model, prompts, answers_length + 1) # (prompt_length + answers_length + 1, batch_size)
            answers_tokens = output[prompt_length:, :] # (answers_length + 1, batch_size), contains tokens
            equality_test = answers_tokens == target_answers # (answers_length + 1, batch_size), contains boolean values
            correct += torch.all(equality_test, axis=0).float().sum()
        accuracy = correct / len(data_test)
    return accuracy.item()

In [66]:
evaluate()

0.0

## Step 5: Train the model, classical approach

### Hyperparameters

In [67]:

epochs = 5
batch_size = 16
learning_rate = 8e-4

reporting_per_epoch = 5
log_interval = len(data_train) // (reporting_per_epoch + 1)
assert(log_interval % batch_size == 0)

In [68]:
def train():
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    best_test_accuracy = None
    test_accuracy = evaluate()
    print('-' * 89)
    print('| initialisation | test accuracy {:5.2f}'.format(test_accuracy))
    print('-' * 89)
    for epoch in range(1, epochs+1):
        epoch_start_time = time.time()
        total_loss = 0.
        start_time = time.time()
        for batch, i in enumerate(range(0, len(data_train) - 1, batch_size)):
            prompts, target_answers, prompt_length, answers_length, _, _ = get_batch("train", i, batch_size)
            prompts = prompts.to(device) # (prompt_length, batch_size)
            target_answers = target_answers.to(device) # (answers_length + 1, batch_size)
            input_tensor = torch.cat((prompts, target_answers), 0) # (prompt_length + answers_length + 1, batch_size)
            model.zero_grad()
            output, _ = model(input_tensor) # (prompt_length + answers_length + 1, batch_size, ntokens)
            output_answers = output[prompt_length-1:-1,:,:].reshape(-1, ntokens) # ((answers_length + 1) * batch_size, ntokens)
            target_answers = target_answers.view(-1)
            loss = F.cross_entropy(output_answers, target_answers)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if i % log_interval == 0 and batch > 0:
                cur_loss = total_loss / log_interval
                elapsed = time.time() - start_time
                print('| {:5d}/{:5d} batches | ms/batch {:5.2f} | loss {:5.2f} | perplexity {:8.2f}'.format(batch, len(data_train) // batch_size,
                                                                                                            elapsed * 1000 / log_interval, cur_loss, math.exp(cur_loss)))
                total_loss = 0
                start_time = time.time()
        test_accuracy = evaluate()
        print('-' * 89)
        print('| end of epoch {:3d} | time: {:5.2f}s | test accuracy {:5.2f}'.format(epoch, (time.time() - epoch_start_time), test_accuracy))
        print('-' * 89)
        # Save the model if the test accuracy is the best we've seen so far.
        if not best_test_accuracy or test_accuracy < best_test_accuracy:
            with open("arithmetic.pt", 'wb') as f:
                torch.save(model, f)
            best_test_accuracy = test_accuracy

In [82]:
train()

-----------------------------------------------------------------------------------------
| initialisation | test accuracy  0.05
-----------------------------------------------------------------------------------------
|   600/ 3600 batches | ms/batch  1.02 | loss  0.03 | perplexity     1.03
|  1200/ 3600 batches | ms/batch  1.00 | loss  0.02 | perplexity     1.02
|  1800/ 3600 batches | ms/batch  1.03 | loss  0.02 | perplexity     1.02
|  2400/ 3600 batches | ms/batch  0.98 | loss  0.02 | perplexity     1.02
|  3000/ 3600 batches | ms/batch  1.00 | loss  0.02 | perplexity     1.02
-----------------------------------------------------------------------------------------
| end of epoch   1 | time: 66.89s | test accuracy  0.56
-----------------------------------------------------------------------------------------
|   600/ 3600 batches | ms/batch  1.03 | loss  0.02 | perplexity     1.02
|  1200/ 3600 batches | ms/batch  0.98 | loss  0.02 | perplexity     1.02
|  1800/ 3600 batches | ms/

KeyboardInterrupt: 

In [70]:
model.eval()

for i in range(20):
    prompt, answers = data_test[i]
    prompt_tensor = torch.tensor(tokenizer.encode(prompt)).view((-1,1))
    output = generate(model, prompt_tensor, len(answers) + 1).view((1,-1))
    print(tokenizer.decode(output.tolist()[0]) + "\t actual result: " + answers)

234+261=489[EOS]	 actual result: 495
758+357=1121[EOS]	 actual result: 1115
717+999=1802[EOS]	 actual result: 1716
856+442=1299[EOS]	 actual result: 1298
312+768=1090[EOS]	 actual result: 1080
333+115=451[EOS]	 actual result: 448
107+332=436[EOS]	 actual result: 439
575+332=909[EOS]	 actual result: 907
855+179=1033[EOS]	 actual result: 1034
204+060=299[EOS]	 actual result: 264
629+746=1370[EOS]	 actual result: 1375
851+150=1010[EOS]	 actual result: 1001
532+099=632[EOS]	 actual result: 631
332+472=829[EOS]	 actual result: 804
971+390=1369[EOS]	 actual result: 1361
041+480=521[EOS]	 actual result: 521
297+528=811[EOS]	 actual result: 825
591+407=999[EOS]	 actual result: 998
776+389=1163[EOS]	 actual result: 1165
081+869=953[EOS]	 actual result: 950


## Step 4bis :  GRPO with POO training

In [71]:
def accuracy_reward(output, answer):
    pattern = r"\[EOS\]"
    output = re.sub(pattern, "", output)
    pattern = r"(\[PAD\])*$"
    output = re.sub(pattern, "", output)
    return 1. if output == answer else 0.

accuracy_reward("123[EOS][PAD][PAD]", "123"), accuracy_reward("123", "124"),

(1.0, 0.0)

In [72]:
def distance_accuracy_reward(output, answer):
    pattern = r"\[EOS\]"
    output = re.sub(pattern, "", output)
    pattern = r"(\[PAD\])*$"
    output = re.sub(pattern, "", output)
    int_output = int(output)
    int_answer = int(answer)
    return abs(int_output - int_answer) / max(int_output, int_answer)

distance_accuracy_reward("123[EOS]", "123"), distance_accuracy_reward("123[PAD]", "124"),

(0.0, 0.008064516129032258)

In [73]:
def digit_accuracy_reward(output, answer):
    pattern = r"\[EOS\]"
    output = re.sub(pattern, "", output)
    pattern = r"(\[PAD\])*$"
    output = re.sub(pattern, "", output)
    return sum(c1 == c2 for (c1,c2) in zip(output, answer)) / max(len(output), len(answer))

digit_accuracy_reward("123[EOS][PAD][PAD]", "123"), digit_accuracy_reward("123[EOS]", "123"),

(1.0, 1.0)

In [74]:
def reward_format(output):
    pattern = r"\d+\[EOS\](\[PAD\])*$"
    return 1. if bool(re.match(pattern, output)) else 0.

reward_format("123[EOS][PAD][PAD]"), reward_format("123[EOS]"), reward_format("123"),

(1.0, 1.0, 0.0)

### Hyperparameters

In [75]:

epochs = 20
batch_size = 16
learning_rate = 1e-4
num_samples = 16
temperature = .8
ppo_clip = 0.2
reporting_per_epoch = 5
log_interval = len(data_train) // (reporting_per_epoch + 1)
assert(log_interval % batch_size == 0)

reward_fun = digit_accuracy_reward
reward_format = reward_format

In [76]:
def compute_rewards(text_outputs, answers):
    repeated_answers = [answer for answer in answers for _ in range(num_samples)]
    rewards = torch.tensor(
        [0.2 * reward_format(output) + 0.8 * reward_fun(output, answer)
         for output, answer in zip(text_outputs, repeated_answers)],
        dtype=torch.float32,
        device=device
    )
    return rewards

In [77]:
def calculate_grpo_advantages(rewards):
    # reshape rewards to group by prompt
    # compute mean and standard deviation for each prompt group
    mean_rewards = rewards.view(-1, num_samples).mean(dim=1)
    std_rewards = rewards.view(-1, num_samples).std(dim=1)
    # expand the means and stds to match the original flat rewards tensor shape
    mean_rewards = mean_rewards.repeat_interleave(num_samples, dim=0)
    std_rewards = std_rewards.repeat_interleave(num_samples, dim=0)
    # normalize rewards to get advantages
    advantages = (rewards - mean_rewards) / (std_rewards + 1e-5)
    return advantages

In [78]:

def compute_log_probs(model, outputs, prompt_length):
    logits, _ = model(outputs)
    # logits.shape = (prompt_length + answers_length + 1, batch_size * num_samples, vocab_size)

    # we only need the log probabilities for the new tokens
    # this introduces a shift: the logits for a position are the predictions for the next token
    logits = logits[prompt_length-1:-1, :, :]
    # logits.shape = (answers_length + 1, batch_size * num_samples, vocab_size)

    # convert raw logits into log probabilities along the vocabulary axis
    log_probs = F.log_softmax(logits, dim=-1)
    # log_probs.shape = (answers_length + 1, batch_size * num_samples, vocab_size)
    return log_probs

In [79]:
def compute_loss_ppo(advantages, old_log_probs, new_log_probs, responses):
    # responses: (T, B)
    responses = responses.unsqueeze(-1)  # (T, B, 1)
    selected_old = old_log_probs.gather(dim=-1, index=responses).squeeze(-1)  # (T, B)
    selected_new = new_log_probs.gather(dim=-1, index=responses).squeeze(-1)  # (T, B)
    ratio = torch.exp(selected_new - selected_old)  # (T, B)
    clipped_ratio = torch.clamp(ratio, 1 - ppo_clip, 1 + ppo_clip)
    advantages = advantages.unsqueeze(0)  # (1, B)
    surrogate1 = ratio * advantages
    surrogate2 = clipped_ratio * advantages
    loss_ppo = -torch.mean(torch.min(surrogate1, surrogate2))
    
    # Indicative calculation of perplexity
    nll = -selected_new.mean()
    ppl = torch.exp(nll)
    return loss_ppo, ppl

def compute_supervised_loss(prompts, target_answers, answers_length, prompt_length):
    # Greedy generation of a complete answer
    greedy_outputs = generate(model, prompts, new_tokens=answers_length + 1, mode="greedy", num_samples=1)
    full_sequence = greedy_outputs  # (prompt_length+answers_length+1, batch_size)
    output, _ = model(full_sequence)
    output_answers = output[prompt_length-1:-1, :]  # (answers_length+1, batch_size, ntokens)
    target = target_answers.view(-1)
    loss_sup = F.cross_entropy(output_answers.reshape(-1, ntokens), target)
    return loss_sup

In [83]:
def train_grpo_ppo(
    # Training hyperparameters with default values
    total_epochs=30,
    batch_size=32,
    learning_rate=3e-4,
    num_samples=16,
    temperature=0.8,
    ppo_epochs=4,
    max_grad_norm=0.8,
    lambda_sup_initial=1.0,
    lambda_sup_final=0.4,
    reporting_per_epoch=5,
    scheduler_step_size=10,
    scheduler_gamma=0.5,
    verbose=False
):
    """
    Train a model using GRPO (Gradient-based Reward Policy Optimization) and PPO (Proximal Policy Optimization).
    
    Args:
        total_epochs: Total number of training epochs
        batch_size: Desired batch size for training
        learning_rate: Initial learning rate for optimizer
        num_samples: Number of samples to generate for reward evaluation
        temperature: Temperature for sampling during generation
        ppo_epochs: Number of PPO epochs per batch
        max_grad_norm: Maximum gradient norm for clipping
        lambda_sup_initial: Initial weight for supervised loss
        lambda_sup_final: Final weight for supervised loss (for linear scheduling)
        reporting_per_epoch: Number of reports per epoch
        scheduler_step_size: Number of epochs before reducing learning rate
        scheduler_gamma: Factor by which to reduce learning rate
        verbose: Whether to print detailed outputs
        
    Returns:
        float: Best test accuracy achieved during training
    """
    # Set up optimizer and learning rate scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, 
        step_size=scheduler_step_size, 
        gamma=scheduler_gamma
    )

    # Initialize tracking variables
    best_test_accuracy = None
    
    # Evaluate model before training
    test_accuracy = evaluate()
    print('-' * 89)
    print('| initialization | test accuracy {:5.2f}'.format(test_accuracy))
    print('-' * 89)
    
    # Calculate reporting intervals
    log_interval = len(data_train) // (reporting_per_epoch + 1)
    
    # Adjust batch_size to ensure log_interval is divisible by batch_size
    # This prevents uneven batches during reporting
    adjusted_batch_size = None
    for bs in range(batch_size, 0, -1):
        if log_interval % bs == 0:
            adjusted_batch_size = bs
            break
    
    # Use adjusted batch size and calculate number of batches
    batch_size = adjusted_batch_size
    num_batches = math.ceil(len(data_train) / batch_size)
    
    # Set model to training mode
    model.train()

    # Main training loop
    for epoch in range(1, total_epochs + 1):
        epoch_start_time = time.time()
        start_time = time.time()
        
        # Calculate supervised loss weight (lambda) for this epoch
        # Linear scheduling from lambda_sup_initial to lambda_sup_final
        lambda_sup = lambda_sup_initial
        if total_epochs > 1:  # Avoid division by zero
            lambda_sup = lambda_sup_initial - (epoch - 1) * ((lambda_sup_initial - lambda_sup_final) / (total_epochs - 1))
        
        # Iterate through batches
        for batch, i in enumerate(range(0, len(data_train) - 1, batch_size)):
            # Get batch data
            prompts, target_answers, prompt_length, answers_length, questions, answers = get_batch("train", i, batch_size)
            prompts = prompts.to(device)
            target_answers = target_answers.to(device)
            
            # Generate samples for reward evaluation
            outputs = generate(
                model, 
                prompts, 
                new_tokens=answers_length + 1, 
                mode="sampling", 
                num_samples=num_samples, 
                temperature=temperature
            )
            
            # Decode generated outputs to text
            text_outputs = [tokenizer.decode(outputs[prompt_length:, j].tolist()) for j in range(outputs.size(1))]
            
            # Compute rewards and advantages
            rewards = compute_rewards(text_outputs, answers)
            advantages = calculate_grpo_advantages(rewards)
            
            # Compute log probabilities for PPO
            old_log_probs = compute_log_probs(model, outputs, prompt_length).detach()
            new_log_probs = compute_log_probs(model, outputs, prompt_length)
            responses = outputs[prompt_length:, :]
            
            # Compute losses
            loss_ppo, ppl = compute_loss_ppo(advantages, old_log_probs, new_log_probs, responses)
            loss_sup = compute_supervised_loss(prompts, target_answers, answers_length, prompt_length)
            
            # Combine losses with lambda weighting
            loss_total = loss_ppo + lambda_sup * loss_sup
            
            # Backpropagation and optimization
            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            optimizer.zero_grad()
            
            # Report progress at regular intervals
            if i % log_interval == 0 and batch > 0:
                elapsed = time.time() - start_time
                print('| {:5d}/{:5d} batches | ms/batch {:5.2f} | loss_total {:5.4f} | Perplexity {:5.2f} '.format(
                    batch, num_batches, elapsed * 1000 / log_interval, 
                    loss_total.item(), ppl.item()))
                
                # Print detailed outputs if verbose mode is enabled
                if verbose:
                    print("\nquestion:", questions[0],
                          "\nanswer:", answers[0],
                          "\noutput:", text_outputs[:num_samples],
                          "\nreward:", rewards[:num_samples],
                          "\nadvantage:", advantages[:num_samples], "\n")
                
                start_time = time.time()
        
        # Evaluate model after each epoch
        test_accuracy = evaluate()
        epoch_time = time.time() - epoch_start_time
        
        print('-' * 89)
        print('| end of epoch {:3d} | time: {:5.2f}s | test accuracy {:5.2f}'.format(
            epoch, epoch_time, test_accuracy))
        print('-' * 89)
        
        # Save model if it's the best so far
        if best_test_accuracy is None or test_accuracy > best_test_accuracy:
            torch.save(model, "arithmetic_grpo_ppo.pt")
            best_test_accuracy = test_accuracy
        
        # Early stopping if perfect accuracy is achieved
        if test_accuracy == 1: 
            break
            
        # Update learning rate scheduler
        scheduler.step()
    
    print("Training complete; best test accuracy: {:5.2f}".format(best_test_accuracy))
    
   

In [84]:
train_grpo_ppo()

-----------------------------------------------------------------------------------------
| initialisation | test accuracy  0.89
-----------------------------------------------------------------------------------------
|   300/ 1800 batches | ms/batch  6.76 | loss_total 18.9814 | Perplexity  3.75 
|   600/ 1800 batches | ms/batch  6.72 | loss_total 26.4031 | Perplexity  7.89 
|   900/ 1800 batches | ms/batch  6.72 | loss_total 3.8362 | Perplexity  9.21 
|  1200/ 1800 batches | ms/batch  6.72 | loss_total 3.8468 | Perplexity  6.92 
|  1500/ 1800 batches | ms/batch  6.73 | loss_total 4.2939 | Perplexity  6.82 
-----------------------------------------------------------------------------------------
| end of epoch   1 | time: 396.75s | test accuracy  0.00
-----------------------------------------------------------------------------------------
|   300/ 1800 batches | ms/batch  6.15 | loss_total 0.2448 | Perplexity  1.24 
|   600/ 1800 batches | ms/batch  6.13 | loss_total 0.0770 | Perplex

KeyboardInterrupt: 

In [85]:
model.eval()

for i in range(20):
    prompt, answers = data_test[i]
    prompt_tensor = torch.tensor(tokenizer.encode(prompt)).view((-1,1))
    output = generate(model, prompt_tensor, len(answers) + 1).view((1,-1))
    print(tokenizer.decode(output.tolist()[0]) + "\t actual result: " + answers)

234+261=495[EOS]	 actual result: 495
758+357=1115[EOS]	 actual result: 1115
717+999=1716[EOS]	 actual result: 1716
856+442=1298[EOS]	 actual result: 1298
312+768=1080[EOS]	 actual result: 1080
333+115=448[EOS]	 actual result: 448
107+332=439[EOS]	 actual result: 439
575+332=907[EOS]	 actual result: 907
855+179=1034[EOS]	 actual result: 1034
204+060=264[EOS]	 actual result: 264
629+746=1375[EOS]	 actual result: 1375
851+150=1001[EOS]	 actual result: 1001
532+099=631[EOS]	 actual result: 631
332+472=804[EOS]	 actual result: 804
971+390=1361[EOS]	 actual result: 1361
041+480=521[EOS]	 actual result: 521
297+528=825[EOS]	 actual result: 825
591+407=998[EOS]	 actual result: 998
776+389=1165[EOS]	 actual result: 1165
081+869=950[EOS]	 actual result: 950
